In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
required_columns = {
    "stop_id","stop_code","stop_name","stop_lat","stop_lat","zone_id", "location_type","parent_station"
}

missing_columns = required_columns - set(bronze_gtfs_stops.columns)
if missing_columns:
    raise ValueError(
        "GRESKA: Izvorni GTFS stops je promenio strukturu, postoje nedostajuce kolone!"
        )


In [0]:
bronze_gtfs_stops = spark.table("bg_traffic.bg_traffic_bronze.gtfs_stops")
bronze_gtfs_stops.display()

In [0]:
bronze_gtfs_stops.printSchema()

In [0]:
bronze_gtfs_stops.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in bronze_gtfs_stops.columns]).show()

### Casting

In [0]:
typed_stops = bronze_gtfs_stops.select(
    F.col("stop_id").cast("integer"),
    F.col("stop_code").cast("integer"),
    F.col("stop_name").cast("string"),
    F.col("stop_lat").cast("double").alias("stop_latitude"),
    F.col("stop_lon").cast("double").alias("stop_longitude"),
    F.col("zone_id").cast("integer"),
    F.col("location_type").cast("integer")
)

typed_stops.printSchema()

### Dedup

In [0]:
dedup_stops = typed_stops.dropDuplicates(["stop_id"])

dedup_count = dedup_stops.count() - typed_stops.count()
print(f"Broj duplikata: {dedup_count}")

### Valid

In [0]:
valid_stops = dedup_stops.filter(
    (F.col("stop_id").isNotNull()) &
    (F.col("stop_latitude").between(44.0, 45.5)) &
    (F.col("stop_longitude").between(19.5,21.0))

).withColumn("silver_processed_at", F.current_timestamp())
valid_stops.display()

In [0]:
if valid_stops.isEmpty():
    raise Exception("GRESKA: Silver tabela za upisivanje je prazna nakon ciscenja!")

### Write in silver table

In [0]:
valid_stops.write.format("delta").mode("overwrite").option(
    "overwriteSchema","true"
).saveAsTable("bg_traffic.bg_traffic_silver.gtfs_stop")